# 02 Event Data Analysis - J03WPY

Notebook ini mengaudit event IDSSE yang relevan terhadap transisi kehilangan bola: `RECOVERY`, `GENERIC:TacklingGame`, dan `GENERIC:PossessionLossBeforeGoal`.

Dua batasan penting:
- kolom event `ball_owning_team` kosong, sehingga possession dan turnover final tidak ditentukan dari event ini;
- konteks event selalu dibatasi pada `period_id` yang sama dan diurutkan berdasarkan timestamp.

Pipeline utama tetap menggunakan perubahan `ball_owning_team_id` pada tracking data. Event digunakan sebagai informasi pendukung dan validasi konteks.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MATCH_ID = 'J03WPY'
EVENTS_PATH = PROJECT_ROOT / 'data' / 'sportec_idsse' / f'{MATCH_ID}_events.csv'
events = pd.read_csv(EVENTS_PATH)
events['timestamp'] = pd.to_timedelta(events['timestamp'], errors='coerce')

print('Events:', EVENTS_PATH)
print('Shape :', events.shape)
display(events[[
    'event_id', 'event_type', 'period_id', 'timestamp', 'team_id',
    'player_id', 'receiver_player_id', 'success',
    'coordinates_x', 'coordinates_y',
]].head())

Events: c:\VAEP-Track\data\sportec_idsse\J03WPY_events.csv
Shape : (1572, 19)


,event_id,event_type,period_id,timestamp,team_id,player_id,receiver_player_id,success,coordinates_x,coordinates_y
0,18237400000006,PASS,1,0 days 00:00:00,DFL-CLU-00000P,DFL-OBJ-002G5J,DFL-OBJ-0028FW,True,0.500000,0.500000
1,18237400000007,PASS,1,0 days 00:00:01.697000,DFL-CLU-00000P,DFL-OBJ-0028FW,DFL-OBJ-002FXT,True,0.718476,0.486765
2,18237400000008,GENERIC:TacklingGame,1,0 days 00:00:04.936000,NaN,NaN,NaN,NaN,0.335905,0.913382
3,18237400000009,PASS,1,0 days 00:00:06.465000,DFL-CLU-00000P,DFL-OBJ-002FXT,NaN,False,0.295810,0.923235
4,18237400000010,PASS,1,0 days 00:00:11.553000,DFL-CLU-000005,DFL-OBJ-0001HW,DFL-OBJ-002G68,True,0.084000,0.680000


In [2]:
inventory = events['event_type'].value_counts().rename('count').to_frame()
team_distribution = events['team_id'].value_counts(
    dropna=False
).rename('count').to_frame()
field_completeness = pd.DataFrame({
    'non_null': events[[
        'timestamp', 'ball_state', 'ball_owning_team', 'team_id',
        'player_id', 'coordinates_x', 'coordinates_y', 'success',
    ]].notna().sum(),
})
field_completeness['percent'] = (
    field_completeness['non_null'] / len(events) * 100
).round(2)

display(inventory)
display(team_distribution)
display(field_completeness)
print(
    'ball_owning_team usable:',
    bool(events['ball_owning_team'].notna().any()),
)

,count
event_type,
PASS,891
GENERIC:OtherBallAction,252
GENERIC:TacklingGame,208
BALL_OUT,70
GENERIC:Delete,60
SHOT,26
FOUL_COMMITTED,18
RECOVERY,12
CARD,9


,count
team_id,
DFL-CLU-00000P,785
DFL-CLU-000005,495
NaN,292


,non_null,percent
timestamp,1572,100.00
ball_state,1572,100.00
ball_owning_team,0,0.00
team_id,1280,81.42
player_id,1278,81.30
coordinates_x,1392,88.55
coordinates_y,1392,88.55
success,916,58.27


ball_owning_team usable: False


In [3]:
TARGET_EVENT_TYPES = [
    'RECOVERY',
    'GENERIC:TacklingGame',
    'GENERIC:PossessionLossBeforeGoal',
]
transition_events = events[
    events['event_type'].isin(TARGET_EVENT_TYPES)
].copy()
transition_summary = transition_events.groupby('event_type').agg(
    count=('event_id', 'size'),
    team_id_available=('team_id', 'count'),
    player_id_available=('player_id', 'count'),
    coordinates_available=('coordinates_x', 'count'),
    success_available=('success', 'count'),
).sort_values('count', ascending=False)
display(transition_summary)

recovery = (
    events[events['event_type'].eq('RECOVERY')]
    .sort_values(['period_id', 'timestamp', 'event_id'])
    .copy()
)
tackles = (
    events[events['event_type'].eq('GENERIC:TacklingGame')]
    .sort_values(['period_id', 'timestamp', 'event_id'])
    .copy()
)
loss_before_goal = events[
    events['event_type'].eq('GENERIC:PossessionLossBeforeGoal')
].copy()

print('Recovery by team:')
display(recovery['team_id'].value_counts().rename('count').to_frame())
display(recovery[[
    'period_id', 'timestamp', 'team_id', 'player_id',
    'coordinates_x', 'coordinates_y',
]])
print('First 15 TacklingGame events:')
display(tackles[[
    'period_id', 'timestamp', 'team_id', 'player_id',
    'coordinates_x', 'coordinates_y',
]].head(15))
print('PossessionLossBeforeGoal events:')
display(loss_before_goal[[
    'period_id', 'timestamp', 'team_id', 'player_id',
    'coordinates_x', 'coordinates_y',
]])

,count,team_id_available,player_id_available,coordinates_available,success_available
event_type,,,,,
GENERIC:TacklingGame,208,0,0,208,0
RECOVERY,12,12,12,12,0
GENERIC:PossessionLossBeforeGoal,1,1,1,1,0


Recovery by team:


,count
team_id,
DFL-CLU-000005,8
DFL-CLU-00000P,4


,period_id,timestamp,team_id,player_id,coordinates_x,coordinates_y
1560,2,-1 days +22:59:06.214000,DFL-CLU-000005,DFL-OBJ-002GG4,0.487619,0.873088
1561,2,-1 days +23:26:28.192000,DFL-CLU-00000P,DFL-OBJ-002GMO,0.758571,0.578235
1562,2,-1 days +23:26:40.856000,DFL-CLU-000005,DFL-OBJ-002G68,0.249048,0.277353
1563,2,-1 days +23:28:18.865000,DFL-CLU-000005,DFL-OBJ-0028T3,0.663143,0.951324
1564,2,-1 days +23:28:50.473000,DFL-CLU-000005,DFL-OBJ-002FYC,0.646000,0.497059
1565,2,0 days 00:12:33.775000,DFL-CLU-000005,DFL-OBJ-002G68,0.756667,0.803824
1566,2,0 days 00:16:24.192000,DFL-CLU-00000P,DFL-OBJ-0000F8,0.676571,0.737500
1567,2,0 days 00:22:21.539000,DFL-CLU-00000P,DFL-OBJ-J01KJ5,0.188952,0.241912
1568,2,0 days 00:40:21.596000,DFL-CLU-000005,DFL-OBJ-0026IA,0.862571,0.822941
1569,2,0 days 00:45:13.598000,DFL-CLU-000005,DFL-OBJ-0001HW,0.966571,0.471912


First 15 TacklingGame events:


,period_id,timestamp,team_id,player_id,coordinates_x,coordinates_y
2,1,0 days 00:00:04.936000,NaN,NaN,0.335905,0.913382
8,1,0 days 00:00:24.490000,NaN,NaN,0.552762,0.162941
16,1,0 days 00:00:26.490000,NaN,NaN,0.464286,0.182059
22,1,0 days 00:00:28.890000,NaN,NaN,0.533810,0.154118
29,1,0 days 00:00:30.730000,NaN,NaN,0.450952,0.331912
12,1,0 days 00:00:31.090000,NaN,NaN,0.432952,0.320147
37,1,0 days 00:00:32.530000,NaN,NaN,0.410095,0.267206
51,1,0 days 00:01:07.410000,NaN,NaN,0.742190,0.625588
54,1,0 days 00:01:08.930000,NaN,NaN,0.749810,0.615294
34,1,0 days 00:01:28.841000,NaN,NaN,0.901048,0.540441


PossessionLossBeforeGoal events:


,period_id,timestamp,team_id,player_id,coordinates_x,coordinates_y
1549,2,0 days 00:01:19.350000,DFL-CLU-00000P,DFL-OBJ-002GMO,0.254952,0.597647


In [4]:
zero_time = pd.Timedelta(0, unit='s')
timestamp_audit = events.groupby('period_id').agg(
    event_count=('event_id', 'size'),
    minimum_timestamp=('timestamp', 'min'),
    maximum_timestamp=('timestamp', 'max'),
)
timestamp_audit['negative_timestamp_count'] = (
    events['timestamp'].lt(zero_time).groupby(events['period_id']).sum()
).astype(int)
display(timestamp_audit)

negative_events = events[events['timestamp'].lt(zero_time)].copy()
print('Negative timestamps by event type:')
display(negative_events.groupby(
    ['period_id', 'event_type']
).size().rename('count').to_frame())

def show_context(event_table, period_id, timestamp, window_seconds=5):
    radius = pd.Timedelta(window_seconds, unit='s')
    start = timestamp - radius
    end = timestamp + radius
    return (
        event_table[
            event_table['period_id'].eq(period_id)
            & event_table['timestamp'].between(start, end)
        ]
        .sort_values(['timestamp', 'event_id'])[[
            'period_id', 'timestamp', 'event_type', 'team_id',
            'player_id', 'receiver_player_id', 'success',
        ]]
    )

valid_recovery = recovery[recovery['timestamp'].ge(zero_time)]
example_recovery = valid_recovery.iloc[0]
print(
    'Context for recovery:', example_recovery['event_id'],
    '| period', int(example_recovery['period_id']),
    '| time', example_recovery['timestamp'],
)
display(show_context(
    events,
    int(example_recovery['period_id']),
    example_recovery['timestamp'],
    window_seconds=5,
))
print(
    'Kesimpulan: RECOVERY berguna sebagai konteks pendukung, tetapi terlalu jarang '
    'dan timestamp negatif perlu ditangani sebelum sinkronisasi event-tracking.'
)

,event_count,minimum_timestamp,maximum_timestamp,negative_timestamp_count
period_id,,,,
1,787,0 days 00:00:00,0 days 00:45:45.046000,0
2,785,-1 days +22:59:06.214000,0 days 00:51:09.786000,13


Negative timestamps by event type:


count
period_id event_type                      
2         CARD                           1
          GENERIC:BallDeflection         1
          GENERIC:PenaltyNotAwarded      1
          GENERIC:RefereeBall            2
          GENERIC:Run                    1
          GENERIC:SpectacularPlay        1
          PASS                           1
          RECOVERY                       5

Context for recovery: 18237403500940 | period 2 | time 0 days 00:12:33.775000


,period_id,timestamp,event_type,team_id,player_id,receiver_player_id,success
968,2,0 days 00:12:29.902000,GENERIC:TacklingGame,NaN,NaN,NaN,NaN
1565,2,0 days 00:12:33.775000,RECOVERY,DFL-CLU-000005,DFL-OBJ-002G68,NaN,NaN
967,2,0 days 00:12:33.776000,PASS,DFL-CLU-000005,DFL-OBJ-002G68,DFL-OBJ-002595,True


Kesimpulan: RECOVERY berguna sebagai konteks pendukung, tetapi terlalu jarang dan timestamp negatif perlu ditangani sebelum sinkronisasi event-tracking.
